In [25]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

In [ ]:
# Load with MultiIndex headers (two rows of headers)
# When loading the pickle, check shape and head immediately
with open('../data/processed/seera_cleaned.pkl', 'rb') as f:
    df_clean = pd.read_pickle(f)

print("Shape:", df_clean.shape)
print("Column levels:", df_clean.columns.nlevels)
print("Head:\n", df_clean.head(2))

print("Dataset loaded successfully!")
print(f"Shape: {df_clean.shape} rows x {df_clean.shape[1]} columns")

<h2>Spliting the dataset </h2>

In [7]:
# ============================================================
# 1. SANITIZE COLUMN NAMES
# ============================================================

if isinstance(df_clean.columns, pd.MultiIndex):
    df_clean.columns = [
        "_".join(str(x).strip() for x in col if str(x).strip() != "")
        for col in df_clean.columns
    ]

df_clean.columns = (
    df_clean.columns
    .astype(str)
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Remove duplicate columns
df_clean = df_clean.loc[
    :, ~df_clean.columns.duplicated()
].copy()

In [ ]:

# ============================================================
# 2. DEFINE TARGET, GROUP, AND ID
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'

# Your cleaned dataset does not contain ProjID,
# so we do not require it.
id_col = 'General Information_ProjID'


# Check required columns
if target_col not in df_clean.columns:
    raise KeyError(
        f"Target column not found: {target_col}"
    )

if group_col not in df_clean.columns:
    raise KeyError(
        f"Organization column not found: {group_col}"
    )

In [ ]:
# ============================================================
# 3. TARGET
# ============================================================

y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)

# Remove rows where target is missing
valid_rows = y.notna()

df_clean = df_clean.loc[valid_rows].copy()
y = y.loc[valid_rows].reset_index(drop=True)

# Log-transform target
y_log = np.log1p(y)

In [ ]:
# ============================================================
# 4. GROUPS
# ============================================================

groups = df_clean[group_col].reset_index(drop=True)

# Make sure groups contain no missing values
if groups.isna().any():
    raise ValueError(
        "Organization ID contains missing values."
    )

In [ ]:

# ============================================================
# 5. BUILD FEATURE MATRIX
# ============================================================

cols_to_drop = [
    target_col,
    group_col
]

# Drop project ID only if it actually exists
if id_col in df_clean.columns:
    cols_to_drop.append(id_col)

X = df_clean.drop(
    columns=cols_to_drop
).copy()

In [ ]:
# ============================================================
# 6. DEFINE CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS  expert availability',
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]

# Only keep columns that actually exist
nominal_cat_cols = [
    col for col in nominal_cat_cols
    if col in X.columns
]

In [ ]:

# ============================================================
# 7. CONVERT CATEGORICAL FEATURES
# ============================================================

for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
    )


In [ ]:
# ============================================================
# 8. CONVERT NUMERIC / ORDINAL FEATURES
# ============================================================

numeric_cols = [
    col for col in X.columns
    if col not in nominal_cat_cols
]

for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )


# CatBoost can handle numeric NaN values.
# But make sure categorical columns contain no NaN.
for col in nominal_cat_cols:

    if X[col].isna().any():
        X[col] = X[col].fillna('Missing')

In [ ]:
# ============================================================
# 9. CATBOOST CATEGORICAL COLUMN INDICES
# ============================================================

cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


print("==========================================")
print("DATASET INFORMATION")
print("==========================================")
print(f"Projects:       {len(X)}")
print(f"Features:       {X.shape[1]}")
print(f"Organizations:  {groups.nunique()}")
print(f"Categorical:    {len(nominal_cat_cols)}")
print("==========================================")

In [ ]:
# ============================================================
# 10. HYPERPARAMETER SEARCH SPACE
# ============================================================

param_grid = [

    {
        'depth': 4,
        'learning_rate': 0.03,
        'iterations': 800,
        'l2_leaf_reg': 3
    },

    {
        'depth': 4,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    },

    {
        'depth': 5,
        'learning_rate': 0.03,
        'iterations': 800,
        'l2_leaf_reg': 3
    },

    {
        'depth': 5,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    },

    {
        'depth': 6,
        'learning_rate': 0.03,
        'iterations': 800,
        'l2_leaf_reg': 3
    },

    {
        'depth': 6,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 6,
        'learning_rate': 0.05,
        'iterations': 800,
        'l2_leaf_reg': 10
    },

    {
        'depth': 7,
        'learning_rate': 0.03,
        'iterations': 800,
        'l2_leaf_reg': 10
    },

    {
        'depth': 7,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    },

    {
        'depth': 8,
        'learning_rate': 0.03,
        'iterations': 800,
        'l2_leaf_reg': 10
    }
]



In [ ]:
# ============================================================
# 11. OUTER LOGO-CV
# ============================================================

outer_logo = LeaveOneGroupOut()

all_y_true = []
all_predictions = []

selected_parameters = []

print("\n==========================================")
print("STARTING NESTED LOGO-CV")
print("==========================================")

outer_fold = 0


for train_idx, test_idx in outer_logo.split(
    X,
    y_log,
    groups
):

    outer_fold += 1

    print(
        f"\nOuter Fold {outer_fold}/{groups.nunique()}"
    )

    # --------------------------------------------------------
    # OUTER TRAIN / TEST
    # --------------------------------------------------------

    X_outer_train = X.iloc[train_idx].copy()
    X_outer_test = X.iloc[test_idx].copy()

    y_outer_train = y_log.iloc[train_idx].copy()
    y_outer_test = y_log.iloc[test_idx].copy()

    groups_outer_train = groups.iloc[train_idx].copy()


    # ========================================================
    # 12. INNER LOGO-CV FOR HYPERPARAMETER TUNING
    # ========================================================

    inner_logo = LeaveOneGroupOut()

    best_score = np.inf
    best_params = None


    for params in param_grid:

        inner_true = []
        inner_pred = []


        for inner_train_idx, inner_val_idx in inner_logo.split(
            X_outer_train,
            y_outer_train,
            groups_outer_train
        ):

            X_inner_train = X_outer_train.iloc[
                inner_train_idx
            ]

            X_inner_val = X_outer_train.iloc[
                inner_val_idx
            ]

            y_inner_train = y_outer_train.iloc[
                inner_train_idx
            ]

            y_inner_val = y_outer_train.iloc[
                inner_val_idx
            ]


            # ------------------------------------------------
            # TRAIN MODEL
            # ------------------------------------------------

            model = CatBoostRegressor(

                iterations=params['iterations'],

                learning_rate=params['learning_rate'],

                depth=params['depth'],

                l2_leaf_reg=params['l2_leaf_reg'],

                cat_features=cat_indices,

                loss_function='RMSE',

                random_seed=42,

                verbose=0
            )


            model.fit(
                X_inner_train,
                y_inner_train
            )


            # ------------------------------------------------
            # PREDICT
            # ------------------------------------------------

            pred_log = model.predict(
                X_inner_val
            )

            pred = np.clip(
                np.expm1(pred_log),
                a_min=0,
                a_max=None
            )

            true = np.expm1(
                y_inner_val
            )


            inner_true.extend(true)
            inner_pred.extend(pred)


        # ====================================================
        # CALCULATE INNER MMRE
        # ====================================================

        inner_true = np.array(inner_true)
        inner_pred = np.array(inner_pred)

        inner_mre = (
            np.abs(inner_true - inner_pred)
            / inner_true
        )

        inner_mmre = np.mean(inner_mre)


        # ====================================================
        # KEEP BEST PARAMETERS
        # ====================================================

        if inner_mmre < best_score:

            best_score = inner_mmre
            best_params = params.copy()


In [ ]:
    # ========================================================
    # 13. TRAIN FINAL MODEL FOR THIS OUTER FOLD
    # ========================================================

    print(
        f"Best parameters: {best_params}"
    )

    print(
        f"Inner MMRE: {best_score:.4f}"
    )


    final_model = CatBoostRegressor(

        iterations=best_params['iterations'],

        learning_rate=best_params['learning_rate'],

        depth=best_params['depth'],

        l2_leaf_reg=best_params['l2_leaf_reg'],

        cat_features=cat_indices,

        loss_function='RMSE',

        random_seed=42,

        verbose=0
    )


    final_model.fit(
        X_outer_train,
        y_outer_train
    )


In [11]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score


# ============================================================
# 1. DATA
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'
id_col = 'General Information_ProjID'

y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)

valid_rows = y.notna()

df = df_clean.loc[valid_rows].copy()

y = y.loc[valid_rows].reset_index(drop=True)

groups = (
    df[group_col]
    .reset_index(drop=True)
)


# ============================================================
# 2. FEATURES
# ============================================================

cols_to_drop = [
    target_col,
    group_col
]

if id_col in df.columns:
    cols_to_drop.append(id_col)

X = df.drop(
    columns=cols_to_drop
).copy()


# ============================================================
# 3. CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS  expert availability',
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]

nominal_cat_cols = [
    col for col in nominal_cat_cols
    if col in X.columns
]


# Convert categorical columns
for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(
            r'\.0$',
            '',
            regex=True
        )
    )


# Convert numerical columns
numeric_cols = [
    col for col in X.columns
    if col not in nominal_cat_cols
]

for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )


# CatBoost categorical indices
cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


# ============================================================
# 4. LOG TRANSFORMATION
# ============================================================

y_log = np.log1p(y)


# ============================================================
# 5. HYPERPARAMETER CONFIGURATIONS
# ============================================================

param_grid = [

    {
        'depth': 4,
        'learning_rate': 0.03,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 5,
        'learning_rate': 0.03,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 6,
        'learning_rate': 0.03,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 7,
        'learning_rate': 0.03,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 6,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 3
    },

    {
        'depth': 6,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    },

    {
        'depth': 5,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    },

    {
        'depth': 7,
        'learning_rate': 0.05,
        'iterations': 500,
        'l2_leaf_reg': 10
    }
]


# ============================================================
# 6. LOGO-CV
# ============================================================

logo = LeaveOneGroupOut()

results = []


# ============================================================
# 7. TEST EACH PARAMETER CONFIGURATION
# ============================================================

for config_number, params in enumerate(
    param_grid,
    start=1
):

    print("\n==========================================")
    print(
        f"Configuration {config_number}"
        f"/{len(param_grid)}"
    )
    print(params)
    print("==========================================")


    all_true = []
    all_pred = []


    for fold, (train_idx, test_idx) in enumerate(
        logo.split(X, y_log, groups),
        start=1
    ):

        print(
            f"Fold {fold}/{groups.nunique()}",
            end="\r"
        )


        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y_log.iloc[train_idx]


        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        model = CatBoostRegressor(

            iterations=params['iterations'],

            learning_rate=params['learning_rate'],

            depth=params['depth'],

            l2_leaf_reg=params['l2_leaf_reg'],

            cat_features=cat_indices,

            loss_function='RMSE',

            random_seed=42,

            verbose=0,

            thread_count=-1
        )


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train
        )


        # ----------------------------------------------------
        # PREDICT
        # ----------------------------------------------------

        pred_log = model.predict(
            X_test
        )

        pred = np.clip(
            np.expm1(pred_log),
            a_min=0,
            a_max=None
        )

        true = y.iloc[test_idx]


        all_true.extend(true)
        all_pred.extend(pred)


    # ========================================================
    # METRICS
    # ========================================================

    all_true = np.array(all_true)
    all_pred = np.array(all_pred)

    mre = (
        np.abs(all_true - all_pred)
        / all_true
    )

    mmre = np.mean(mre)

    mdmre = np.median(mre)

    pred25 = np.mean(
        mre <= 0.25
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    r2 = r2_score(
        all_true,
        all_pred
    )


    # ========================================================
    # SAVE RESULT
    # ========================================================

    results.append({

        'depth':
            params['depth'],

        'learning_rate':
            params['learning_rate'],

        'iterations':
            params['iterations'],

        'l2_leaf_reg':
            params['l2_leaf_reg'],

        'MMRE':
            mmre,

        'MdMRE':
            mdmre,

        'PRED25':
            pred25,

        'MAE':
            mae,

        'R2':
            r2
    })


    print(
        f"\nMMRE: {mmre:.4f}"
    )

    print(
        f"MdMRE: {mdmre:.4f}"
    )

    print(
        f"PRED(25): {pred25 * 100:.2f}%"
    )

    print(
        f"MAE: {mae:.2f}"
    )

    print(
        f"R²: {r2:.4f}"
    )


# ============================================================
# 8. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(results)


# Sort by MMRE
results_df = results_df.sort_values(
    by='MMRE'
).reset_index(drop=True)


print("\n==========================================")
print("HYPERPARAMETER TUNING RESULTS")
print("==========================================")

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 9. BEST CONFIGURATION
# ============================================================

best = results_df.iloc[0]

print("\n==========================================")
print("BEST CONFIGURATION")
print("==========================================")

print(
    f"Depth:          {best['depth']}"
)

print(
    f"Learning rate:  {best['learning_rate']}"
)

print(
    f"Iterations:     {best['iterations']}"
)

print(
    f"L2 regularization: {best['l2_leaf_reg']}"
)

print("------------------------------------------")

print(
    f"MMRE:           {best['MMRE']:.4f}"
)

print(
    f"MdMRE:          {best['MdMRE']:.4f}"
)

print(
    f"PRED(25):       {best['PRED25'] * 100:.2f}%"
)

print(
    f"MAE:            {best['MAE']:.2f}"
)

print(
    f"R²:             {best['R2']:.4f}"
)

print("==========================================")


Configuration 1/8
{'depth': 4, 'learning_rate': 0.03, 'iterations': 500, 'l2_leaf_reg': 3}


KeyboardInterrupt: 

In [15]:
import numpy as np
import pandas as pd
import optuna

from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score


# ============================================================
# 1. SETTINGS
# ============================================================

RANDOM_SEED = 42

# Number of Optuna trials
N_TRIALS = 12

# Number of folds ONLY during tuning
TUNING_FOLDS = 5

optuna.logging.set_verbosity(optuna.logging.WARNING)


# ============================================================
# 2. DATA PREPARATION
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'
id_col = 'General Information_ProjID'


# Target
y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)


# Remove rows with missing target
valid_rows = y.notna()

df = df_clean.loc[valid_rows].copy()

y = y.loc[valid_rows].reset_index(drop=True)

groups = (
    df[group_col]
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURES
# ============================================================

cols_to_drop = [
    target_col,
    group_col
]

if id_col in df.columns:
    cols_to_drop.append(id_col)


X = df.drop(
    columns=cols_to_drop
).copy()


# ============================================================
# 4. CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS expert availability',
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]


nominal_cat_cols = [
    col
    for col in nominal_cat_cols
    if col in X.columns
]


# Convert categorical variables to strings
for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(
            r'\.0$',
            '',
            regex=True
        )
    )


# ============================================================
# 5. NUMERIC FEATURES
# ============================================================

numeric_cols = [
    col
    for col in X.columns
    if col not in nominal_cat_cols
]


for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )

    # CatBoost can handle numeric NaN values.
    # We intentionally DO NOT fill them with zero.
    

# CatBoost categorical feature indices
cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


# ============================================================
# 6. TARGET TRANSFORMATION
# ============================================================

y_log = np.log1p(y)


# ============================================================
# 7. GROUP-AWARE CV FOR TUNING
# ============================================================

# IMPORTANT:
# Organizations are kept together.
#
# This prevents projects from the same organization
# appearing in both training and validation folds.

gkf = GroupKFold(
    n_splits=TUNING_FOLDS
)


# ============================================================
# 8. OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    # --------------------------------------------------------
    # Hyperparameters
    # --------------------------------------------------------

    params = {

        # Based on previous experiment:
        # depth 4-7 was tested and depth 5 performed best.
        'depth': trial.suggest_int(
            'depth',
            3,
            6
        ),

        # Focus around the successful 0.03-0.05 region
        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.08,
            log=True
        ),

        # Allow early stopping to determine useful number
        # of iterations.
        'iterations': trial.suggest_int(
            'iterations',
            300,
            1000,
            step=100
        ),

        # Previous best = 10
        'l2_leaf_reg': trial.suggest_float(
            'l2_leaf_reg',
            5,
            20,
            log=True
        ),

        # Additional regularization parameter
        'random_strength': trial.suggest_float(
            'random_strength',
            0.1,
            3.0,
            log=True
        ),

        # Controls Bayesian bootstrap randomness
        'bagging_temperature': trial.suggest_float(
            'bagging_temperature',
            0.5,
            2
        )
    }


    # --------------------------------------------------------
    # Storage for predictions
    # --------------------------------------------------------

    all_true = []
    all_pred = []


    # --------------------------------------------------------
    # Group-aware CV
    # --------------------------------------------------------

    for train_idx, valid_idx in gkf.split(
        X,
        y_log,
        groups
    ):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y_log.iloc[train_idx]
        y_valid = y_log.iloc[valid_idx]


        # ----------------------------------------------------
        # CatBoost
        # ----------------------------------------------------

        model = CatBoostRegressor(

            iterations=params['iterations'],

            learning_rate=params['learning_rate'],

            depth=params['depth'],

            l2_leaf_reg=params['l2_leaf_reg'],

            random_strength=params['random_strength'],

            bagging_temperature=params[
                'bagging_temperature'
            ],

            cat_features=cat_indices,

            loss_function='MAPE',

            random_seed=RANDOM_SEED,

            # Early stopping
            od_type='Iter',

            od_wait=50,

            # Use CPU efficiently
            thread_count=-1,

            verbose=False,

            # Prevent CatBoost from creating files
            allow_writing_files=False
        )


        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train,

            eval_set=(
                X_valid,
                y_valid
            ),

            use_best_model=True
        )


        # ----------------------------------------------------
        # Predict
        # ----------------------------------------------------

        pred_log = model.predict(
            X_valid
        )


        pred = np.clip(
            np.expm1(pred_log),
            a_min=0,
            a_max=None
        )


        true = y.iloc[valid_idx]


        all_true.extend(
            true.to_numpy()
        )

        all_pred.extend(
            pred
        )


    # ========================================================
    # METRICS
    # ========================================================

    all_true = np.array(
        all_true
    )

    all_pred = np.array(
        all_pred
    )


    # MRE
    mre = (
        np.abs(
            all_true - all_pred
        )
        / all_true
    )


    # Main objective
    mmre = np.mean(
        mre
    )


    # Secondary metrics
    mdmre = np.median(
        mre
    )

    pred25 = np.mean(
        mre <= 0.25
    )

    mae = mean_absolute_error(
        all_true,
        all_pred
    )

    r2 = r2_score(
        all_true,
        all_pred
    )


    # Save secondary metrics
    trial.set_user_attr(
        'MdMRE',
        mdmre
    )

    trial.set_user_attr(
        'PRED25',
        pred25
    )

    trial.set_user_attr(
        'MAE',
        mae
    )

    trial.set_user_attr(
        'R2',
        r2
    )


    # Return MMRE
    return mmre


# ============================================================
# 9. CREATE OPTUNA STUDY
# ============================================================

study = optuna.create_study(

    direction='minimize',

    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_SEED
    )
)


# ============================================================
# 10. RUN TUNING
# ============================================================

print(
    f"Starting Optuna tuning:"
)

print(
    f"{N_TRIALS} trials × "
    f"{TUNING_FOLDS} group-aware folds"
)

print(
    "This is ONLY the tuning stage."
)


study.optimize(
    objective,

    n_trials=N_TRIALS,

    show_progress_bar=True
)


# ============================================================
# 11. BEST RESULT
# ============================================================

best_trial = study.best_trial


print("\n")
print("=" * 55)
print("        BEST OPTUNA CONFIGURATION")
print("=" * 55)


print(
    f"Trial:       {best_trial.number}"
)

print(
    f"MMRE:        {best_trial.value:.4f}"
)


print("-" * 55)


for param, value in best_trial.params.items():

    print(
        f"{param:<22}: {value}"
    )


print("-" * 55)


print(
    f"MdMRE:       "
    f"{best_trial.user_attrs['MdMRE']:.4f}"
)

print(
    f"PRED(25):    "
    f"{best_trial.user_attrs['PRED25'] * 100:.2f}%"
)

print(
    f"MAE:         "
    f"{best_trial.user_attrs['MAE']:.2f}"
)

print(
    f"R²:          "
    f"{best_trial.user_attrs['R2']:.4f}"
)


print("=" * 55)

Starting Optuna tuning:
12 trials × 5 group-aware folds
This is ONLY the tuning stage.


  0%|          | 0/12 [00:00<?, ?it/s]



        BEST OPTUNA CONFIGURATION
Trial:       11
MMRE:        0.5113
-------------------------------------------------------
depth                 : 3
learning_rate         : 0.04124173274946173
iterations            : 600
l2_leaf_reg           : 13.500717070418117
random_strength       : 0.41882193959396974
bagging_temperature   : 1.9187145799605694
-------------------------------------------------------
MdMRE:       0.3415
PRED(25):    42.02%
MAE:         3994.96
R²:          0.4325


<h2>Final trial: full LOGO with parameters optained from optuna </h2>

In [16]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score


# ============================================================
# FINAL CATBOOST EVALUATION
# FROZEN OPTUNA CONFIGURATION + FULL LOGO-CV
# ============================================================

RANDOM_SEED = 42


# ============================================================
# 1. COLUMN DEFINITIONS
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'
id_col = 'General Information_ProjID'


# ============================================================
# 2. DATA PREPARATION
# ============================================================

# Convert target to numeric
y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)

# Remove rows where target is missing
valid_rows = y.notna()

df = df_clean.loc[valid_rows].copy()

y = y.loc[valid_rows].reset_index(drop=True)

groups = (
    df[group_col]
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURES
# ============================================================

# Target and organization ID must NOT be predictors.
# Project ID is also excluded if present.

cols_to_drop = [
    target_col,
    group_col
]

if id_col in df.columns:
    cols_to_drop.append(id_col)

X = df.drop(
    columns=cols_to_drop
).copy()


# ============================================================
# 4. CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS expert availability',
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]

# Keep only categorical columns that actually exist
nominal_cat_cols = [
    col
    for col in nominal_cat_cols
    if col in X.columns
]


# Convert categorical values to strings
for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(
            r'\.0$',
            '',
            regex=True
        )
    )


# ============================================================
# 5. NUMERICAL FEATURES
# ============================================================

numeric_cols = [
    col
    for col in X.columns
    if col not in nominal_cat_cols
]

for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )


# CatBoost categorical feature indices
cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


# ============================================================
# 6. TARGET TRANSFORMATION
# ============================================================

# Same transformation used during tuning
y_log = np.log1p(y)


# ============================================================
# 7. LOGO VALIDATION SETUP
# ============================================================

logo = LeaveOneGroupOut()

n_groups = groups.nunique()


print("=" * 65)
print("        FINAL CATBOOST LOGO EVALUATION")
print("=" * 65)

print(f"Observations:       {len(X)}")
print(f"Organizations:      {n_groups}")
print(f"Features:           {X.shape[1]}")
print(f"Categorical:        {len(cat_indices)}")
print("Validation:         Leave-One-Organization-Out")
print(f"Expected folds:     {n_groups}")


# ============================================================
# 8. FROZEN OPTUNA CONFIGURATION
# ============================================================

# Trial 11 — BEST OPTUNA CONFIGURATION

final_params = {
    'depth': 3,
    'learning_rate': 0.04124173274946173,
    'iterations': 600,
    'l2_leaf_reg': 13.500717070418117,
    'random_strength': 0.41882193959396974,
    'bagging_temperature': 1.9187145799605694
}


print("\n" + "-" * 65)
print("FROZEN OPTUNA PARAMETERS")
print("-" * 65)

for name, value in final_params.items():
    print(f"{name:<22}: {value}")


# ============================================================
# 9. STORAGE FOR OUT-OF-FOLD PREDICTIONS
# ============================================================

all_true = []
all_pred = []

prediction_records = []


# ============================================================
# 10. FULL LOGO CROSS-VALIDATION
# ============================================================

print("\n")
print("Starting final LOGO evaluation...")
print()


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y_log, groups),
    start=1
):

    # --------------------------------------------------------
    # Split data
    # --------------------------------------------------------

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_log = y_log.iloc[train_idx]

    true = y.iloc[test_idx].to_numpy()


    # --------------------------------------------------------
    # Create CatBoost model
    # --------------------------------------------------------

    model = CatBoostRegressor(

        # FROZEN OPTUNA PARAMETERS
        depth=final_params['depth'],
        learning_rate=final_params['learning_rate'],
        iterations=final_params['iterations'],
        l2_leaf_reg=final_params['l2_leaf_reg'],
        random_strength=final_params['random_strength'],
        bagging_temperature=final_params['bagging_temperature'],

        # Model settings
        cat_features=cat_indices,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,

        # Use all available CPU threads
        thread_count=-1,

        # No training output
        verbose=False,

        # Prevent CatBoost from creating extra files
        allow_writing_files=False
    )


    # --------------------------------------------------------
    # Train on all organizations EXCEPT the held-out one
    # --------------------------------------------------------

    model.fit(
        X_train,
        y_train_log
    )


    # --------------------------------------------------------
    # Predict the held-out organization
    # --------------------------------------------------------

    pred_log = model.predict(
        X_test
    )


    # Convert prediction back from log scale
    pred = np.clip(
        np.expm1(pred_log),
        a_min=0,
        a_max=None
    )


    # --------------------------------------------------------
    # Store predictions
    # --------------------------------------------------------

    all_true.extend(true)
    all_pred.extend(pred)


    # --------------------------------------------------------
    # Store detailed project-level results
    # --------------------------------------------------------

    for i, original_idx in enumerate(test_idx):

        actual = true[i]
        predicted = pred[i]

        absolute_error = abs(
            actual - predicted
        )

        mre_value = (
            absolute_error / actual
        )

        prediction_records.append({
            'Fold': fold,
            'Organization_ID': groups.iloc[original_idx],
            'Actual_Effort': actual,
            'Predicted_Effort': predicted,
            'Absolute_Error': absolute_error,
            'MRE': mre_value
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Fold {fold:>2}/{n_groups} completed"
    )


# ============================================================
# 11. CONVERT ALL PREDICTIONS TO ARRAYS
# ============================================================

y_true_arr = np.array(
    all_true
)

y_pred_arr = np.array(
    all_pred
)


# ============================================================
# 12. CALCULATE MRE
# ============================================================

mre = (
    np.abs(
        y_true_arr - y_pred_arr
    )
    / y_true_arr
)


# ============================================================
# 13. FINAL EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(
    y_true_arr,
    y_pred_arr
)

r2 = r2_score(
    y_true_arr,
    y_pred_arr
)

mmre = np.mean(
    mre
)

mdmre = np.median(
    mre
)

pred25 = np.mean(
    mre <= 0.25
)


# ============================================================
# 14. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("       FINAL CATBOOST LOGO-CV RESULTS")
print("=" * 65)

print(
    f"MAE:       {mae:.2f} hours"
)

print(
    f"R²:        {r2:.4f}"
)

print(
    f"MMRE:      {mmre:.4f} ({mmre * 100:.2f}%)"
)

print(
    f"MdMRE:     {mdmre:.4f} ({mdmre * 100:.2f}%)"
)

print(
    f"PRED(25):  {pred25 * 100:.2f}%"
)

print("=" * 65)


# ============================================================
# 15. PROJECT-LEVEL PREDICTION TABLE
# ============================================================

predictions_df = pd.DataFrame(
    prediction_records
)


print("\nProject-level predictions created.")
print(
    f"Number of predictions: {len(predictions_df)}"
)

print("\nFirst 10 predictions:")
print(
    predictions_df.head(10)
)

        FINAL CATBOOST LOGO EVALUATION
Observations:       119
Organizations:      41
Features:           50
Categorical:        6
Validation:         Leave-One-Organization-Out
Expected folds:     41

-----------------------------------------------------------------
FROZEN OPTUNA PARAMETERS
-----------------------------------------------------------------
depth                 : 3
learning_rate         : 0.04124173274946173
iterations            : 600
l2_leaf_reg           : 13.500717070418117
random_strength       : 0.41882193959396974
bagging_temperature   : 1.9187145799605694


Starting final LOGO evaluation...

Fold  1/41 completed
Fold  2/41 completed
Fold  3/41 completed
Fold  4/41 completed
Fold  5/41 completed
Fold  6/41 completed
Fold  7/41 completed
Fold  8/41 completed
Fold  9/41 completed
Fold 10/41 completed
Fold 11/41 completed
Fold 12/41 completed
Fold 13/41 completed
Fold 14/41 completed
Fold 15/41 completed
Fold 16/41 completed
Fold 17/41 completed
Fold 18/41 complete

<h2>Error anlaysis: </h2>

In [17]:

# ============================================================
# ERROR ANALYSIS — FINAL CATBOOST MODEL
# Uses predictions_df from the final 41-fold LOGO evaluation
# ============================================================


# ------------------------------------------------------------
# 1. BASIC ERROR SUMMARY
# ------------------------------------------------------------

error_df = predictions_df.copy()

print("=" * 70)
print("                 ERROR ANALYSIS")
print("=" * 70)

print(f"Number of predictions: {len(error_df)}")


# ============================================================
# 2. ERROR CATEGORIES
# ============================================================

# Signed error:
# Positive = model OVERestimated
# Negative = model UNDERestimated

error_df['Signed_Error'] = (
    error_df['Predicted_Effort']
    - error_df['Actual_Effort']
)

# Percentage error
error_df['Percentage_Error'] = (
    error_df['Signed_Error']
    / error_df['Actual_Effort']
    * 100
)

# Absolute Percentage Error
error_df['Absolute_Percentage_Error'] = (
    error_df['MRE'] * 100
)


# ============================================================
# 3. OVERALL ERROR STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("OVERALL ERROR STATISTICS")
print("=" * 70)

print(
    f"Mean Absolute Error:       "
    f"{error_df['Absolute_Error'].mean():.2f} hours"
)

print(
    f"Median Absolute Error:     "
    f"{error_df['Absolute_Error'].median():.2f} hours"
)

print(
    f"Mean MRE:                  "
    f"{error_df['MRE'].mean():.4f}"
)

print(
    f"Median MRE:                "
    f"{error_df['MRE'].median():.4f}"
)

print(
    f"Mean Signed Error:         "
    f"{error_df['Signed_Error'].mean():.2f} hours"
)

print(
    f"Median Signed Error:       "
    f"{error_df['Signed_Error'].median():.2f} hours"
)


# ============================================================
# 4. OVERALL DIRECTION OF ERRORS
# ============================================================

overestimated = (
    error_df['Signed_Error'] > 0
).sum()

underestimated = (
    error_df['Signed_Error'] < 0
).sum()

exact = (
    error_df['Signed_Error'] == 0
).sum()

print("\n" + "=" * 70)
print("ERROR DIRECTION")
print("=" * 70)

print(
    f"Overestimated projects:    "
    f"{overestimated} "
    f"({overestimated / len(error_df) * 100:.2f}%)"
)

print(
    f"Underestimated projects:   "
    f"{underestimated} "
    f"({underestimated / len(error_df) * 100:.2f}%)"
)

print(
    f"Exact predictions:         "
    f"{exact}"
)


# ============================================================
# 5. TOP 10 WORST PREDICTIONS
# ============================================================

worst_predictions = (
    error_df
    .sort_values(
        'MRE',
        ascending=False
    )
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 WORST PREDICTIONS — BY MRE")
print("=" * 70)

print(
    worst_predictions[
        [
            'Fold',
            'Organization_ID',
            'Actual_Effort',
            'Predicted_Effort',
            'Absolute_Error',
            'MRE'
        ]
    ].to_string(index=False)
)


# ============================================================
# 6. TOP 10 LARGEST ABSOLUTE ERRORS
# ============================================================

largest_absolute_errors = (
    error_df
    .sort_values(
        'Absolute_Error',
        ascending=False
    )
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 LARGEST ABSOLUTE ERRORS")
print("=" * 70)

print(
    largest_absolute_errors[
        [
            'Fold',
            'Organization_ID',
            'Actual_Effort',
            'Predicted_Effort',
            'Absolute_Error',
            'MRE'
        ]
    ].to_string(index=False)
)


# ============================================================
# 7. BEST PREDICTIONS
# ============================================================

best_predictions = (
    error_df
    .sort_values(
        'MRE',
        ascending=True
    )
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 BEST PREDICTIONS — LOWEST MRE")
print("=" * 70)

print(
    best_predictions[
        [
            'Fold',
            'Organization_ID',
            'Actual_Effort',
            'Predicted_Effort',
            'Absolute_Error',
            'MRE'
        ]
    ].to_string(index=False)
)


# ============================================================
# 8. PREDICTION ACCURACY BANDS
# ============================================================

within_10 = (
    error_df['MRE'] <= 0.10
).sum()

within_25 = (
    error_df['MRE'] <= 0.25
).sum()

within_50 = (
    error_df['MRE'] <= 0.50
).sum()

over_50 = (
    error_df['MRE'] > 0.50
).sum()

over_100 = (
    error_df['MRE'] > 1.00
).sum()

print("\n" + "=" * 70)
print("PREDICTION ACCURACY BANDS")
print("=" * 70)

print(
    f"Within ±10%:               "
    f"{within_10}/{len(error_df)} "
    f"({within_10 / len(error_df) * 100:.2f}%)"
)

print(
    f"Within ±25%:               "
    f"{within_25}/{len(error_df)} "
    f"({within_25 / len(error_df) * 100:.2f}%)"
)

print(
    f"Within ±50%:               "
    f"{within_50}/{len(error_df)} "
    f"({within_50 / len(error_df) * 100:.2f}%)"
)

print(
    f"Error > 50%:               "
    f"{over_50}/{len(error_df)} "
    f"({over_50 / len(error_df) * 100:.2f}%)"
)

print(
    f"Error > 100%:              "
    f"{over_100}/{len(error_df)} "
    f"({over_100 / len(error_df) * 100:.2f}%)"
)


# ============================================================
# 9. ORGANIZATION-LEVEL ERROR ANALYSIS
# ============================================================

organization_analysis = (
    error_df
    .groupby('Organization_ID')
    .agg(
        Projects=('Actual_Effort', 'count'),
        Mean_Actual=('Actual_Effort', 'mean'),
        Mean_Predicted=('Predicted_Effort', 'mean'),
        MAE=('Absolute_Error', 'mean'),
        MMRE=('MRE', 'mean'),
        MdMRE=('MRE', 'median'),
        PRED25=('MRE', lambda x: (x <= 0.25).mean())
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("ORGANIZATION-LEVEL ERROR ANALYSIS")
print("=" * 70)

print(
    organization_analysis
    .sort_values('MMRE', ascending=False)
    .to_string(index=False)
)


# ============================================================
# 10. ACTUAL EFFORT SIZE BANDS
# ============================================================

# Divide projects according to actual effort.
error_df['Effort_Size'] = pd.cut(
    error_df['Actual_Effort'],
    bins=[
        -np.inf,
        2000,
        5000,
        10000,
        np.inf
    ],
    labels=[
        '<= 2,000 hours',
        '2,001–5,000 hours',
        '5,001–10,000 hours',
        '> 10,000 hours'
    ]
)

size_analysis = (
    error_df
    .groupby(
        'Effort_Size',
        observed=True
    )
    .agg(
        Projects=('Actual_Effort', 'count'),
        Mean_Actual=('Actual_Effort', 'mean'),
        Mean_Predicted=('Predicted_Effort', 'mean'),
        MAE=('Absolute_Error', 'mean'),
        MMRE=('MRE', 'mean'),
        MdMRE=('MRE', 'median'),
        PRED25=('MRE', lambda x: (x <= 0.25).mean())
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("ERROR BY ACTUAL EFFORT SIZE")
print("=" * 70)

print(
    size_analysis.to_string(index=False)
)


# ============================================================
# 11. UNDER/OVER ESTIMATION SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("UNDER/OVER ESTIMATION")
print("=" * 70)

print(
    f"Average signed error: "
    f"{error_df['Signed_Error'].mean():.2f} hours"
)

if error_df['Signed_Error'].mean() > 0:

    print(
        "Overall tendency: MODEL TENDS TO OVERESTIMATE"
    )

elif error_df['Signed_Error'].mean() < 0:

    print(
        "Overall tendency: MODEL TENDS TO UNDERESTIMATE"
    )

else:

    print(
        "Overall tendency: NO OVERALL BIAS"
    )


# ============================================================
# 12. SAVE COMPLETE ERROR ANALYSIS TABLE
# ============================================================

error_analysis_df = error_df.sort_values(
    'MRE',
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("ERROR ANALYSIS COMPLETE")
print("=" * 70)

print(
    "Available dataframe: error_analysis_df"
)

                 ERROR ANALYSIS
Number of predictions: 119

OVERALL ERROR STATISTICS
Mean Absolute Error:       4008.54 hours
Median Absolute Error:     1549.09 hours
Mean MRE:                  0.5477
Median MRE:                0.3304
Mean Signed Error:         -1880.99 hours
Median Signed Error:       86.41 hours

ERROR DIRECTION
Overestimated projects:    66 (55.46%)
Underestimated projects:   53 (44.54%)
Exact predictions:         0

TOP 10 WORST PREDICTIONS — BY MRE
 Fold  Organization_ID  Actual_Effort  Predicted_Effort  Absolute_Error      MRE
   28               29          704.0       4606.689172     3902.689172 5.543593
   11               11          924.0       4681.725780     3757.725780 4.066803
   11               11          924.0       4313.524954     3389.524954 3.668317
   13               13         1232.0       4165.569707     2933.569707 2.381144
   26               27         1936.0       6211.606355     4275.606355 2.208474
    5                5          577.5  

In [18]:
# Quick sanity check
test_val = y.iloc[0]
print(f"Original: {test_val}")
print(f"After log1p then expm1: {np.expm1(np.log1p(test_val))}")

Original: 3168.0
After log1p then expm1: 3167.999999999999


In [19]:
# ==========================================
# INVESTIGATE PROJECTS WITH MRE > 100%
# ==========================================

high_mre = error_analysis_df[error_analysis_df['MRE'] > 1.0].copy()
high_mre = high_mre.sort_values('MRE', ascending=False)

print("======================================================================")
print("         PROJECTS WITH MRE > 100%")
print("======================================================================")
print(f"Count: {len(high_mre)} projects\n")

print(high_mre[[
    'Fold', 'Organization_ID', 'Actual_Effort', 
    'Predicted_Effort', 'Absolute_Error', 'MRE'
]].to_string(index=False))

# ==========================================
# EFFORT SIZE DISTRIBUTION OF HIGH MRE
# ==========================================
print("\n======================================================================")
print("         EFFORT SIZE — ARE HIGH MRE PROJECTS SMALL?")
print("======================================================================")
bins   = [0, 1000, 2000, 5000, 10000, np.inf]
labels = ['<=1000', '1001-2000', '2001-5000', '5001-10000', '>10000']

high_mre['effort_band'] = pd.cut(
    high_mre['Actual_Effort'], bins=bins, labels=labels
)
print(high_mre['effort_band'].value_counts().sort_index())

# ==========================================
# ORGANIZATION PROFILE OF HIGH MRE
# ==========================================
print("\n======================================================================")
print("         WHICH ORGANIZATIONS APPEAR IN HIGH MRE?")
print("======================================================================")
org_counts = high_mre['Organization_ID'].value_counts()
print(org_counts)

print(f"\nUnique organizations affected: {high_mre['Organization_ID'].nunique()}")
print(f"Organization IDs: {sorted(high_mre['Organization_ID'].unique())}")

# ==========================================
# COMPARE HIGH MRE vs REST
# ==========================================
print("\n======================================================================")
print("         HIGH MRE vs REST — FEATURE COMPARISON")
print("======================================================================")

low_mre = error_analysis_df[error_analysis_df['MRE'] <= 1.0]

comparison = pd.DataFrame({
    'High_MRE_projects': [
        len(high_mre),
        high_mre['Actual_Effort'].mean(),
        high_mre['Actual_Effort'].median(),
        high_mre['Predicted_Effort'].mean(),
        high_mre['MRE'].mean(),
    ],
    'Rest_of_projects': [
        len(low_mre),
        low_mre['Actual_Effort'].mean(),
        low_mre['Actual_Effort'].median(),
        low_mre['Predicted_Effort'].mean(),
        low_mre['MRE'].mean(),
    ]
}, index=[
    'Count',
    'Mean Actual Effort',
    'Median Actual Effort',
    'Mean Predicted Effort',
    'Mean MRE'
])

print(comparison.round(2))

# ==========================================
# DIRECTION OF ERROR — ARE THEY ALL OVERESTIMATED?
# ==========================================
print("\n======================================================================")
print("         ERROR DIRECTION FOR HIGH MRE PROJECTS")
print("======================================================================")

high_mre['direction'] = np.where(
    high_mre['Predicted_Effort'] > high_mre['Actual_Effort'],
    'Overestimated',
    'Underestimated'
)

print(high_mre['direction'].value_counts())
print(f"\nMean actual effort:    {high_mre['Actual_Effort'].mean():.0f} hours")
print(f"Mean predicted effort: {high_mre['Predicted_Effort'].mean():.0f} hours")
print(f"Mean absolute error:   {high_mre['Absolute_Error'].mean():.0f} hours")

# ==========================================
# FOLD DISTRIBUTION — ANY PATTERN?
# ==========================================
print("\n======================================================================")
print("         FOLD DISTRIBUTION OF HIGH MRE PROJECTS")
print("======================================================================")
print(high_mre[['Fold', 'Organization_ID', 'Actual_Effort', 
                 'Predicted_Effort', 'MRE', 'direction']].to_string(index=False))

         PROJECTS WITH MRE > 100%
Count: 12 projects

 Fold  Organization_ID  Actual_Effort  Predicted_Effort  Absolute_Error      MRE
   28               29          704.0       4606.689172     3902.689172 5.543593
   11               11          924.0       4681.725780     3757.725780 4.066803
   11               11          924.0       4313.524954     3389.524954 3.668317
   13               13         1232.0       4165.569707     2933.569707 2.381144
   26               27         1936.0       6211.606355     4275.606355 2.208474
    5                5          577.5       1620.394917     1042.894917 1.805879
   41               45         2640.0       7005.362077     4365.362077 1.653546
   31               32          704.0       1811.024632     1107.024632 1.572478
    5                5         1584.0       3688.202283     2104.202283 1.328411
    5                5          528.0       1214.833728      686.833728 1.300821
    6                6         5346.0      11611.381888

<h2> Rerunning the model without the 12 peojects 

In [20]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import mannwhitneyu


# ============================================================
# FINAL CATBOOST EVALUATION
# FROZEN OPTUNA CONFIGURATION + FULL LOGO-CV
# ============================================================

RANDOM_SEED = 42


# ============================================================
# 1. COLUMN DEFINITIONS
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'
id_col = 'General Information_ProjID'


# ============================================================
# 2. DATA PREPARATION
# ============================================================

# Convert target to numeric
y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)

# Remove rows where target is missing
valid_rows = y.notna()

df = df_clean.loc[valid_rows].copy()

y = y.loc[valid_rows].reset_index(drop=True)

groups = (
    df[group_col]
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURES
# ============================================================

# Target and organization ID must NOT be predictors.
# Project ID is also excluded if present.

cols_to_drop = [
    target_col,
    group_col
]

if id_col in df.columns:
    cols_to_drop.append(id_col)

X = df.drop(
    columns=cols_to_drop
).copy()


# ============================================================
# 4. CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS expert availability',
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]

# Keep only categorical columns that actually exist
nominal_cat_cols = [
    col
    for col in nominal_cat_cols
    if col in X.columns
]


# Convert categorical values to strings
for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(
            r'\.0$',
            '',
            regex=True
        )
    )


# ============================================================
# 5. NUMERICAL FEATURES
# ============================================================

numeric_cols = [
    col
    for col in X.columns
    if col not in nominal_cat_cols
]

for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )


# CatBoost categorical feature indices
cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


# ============================================================
# 6. TARGET TRANSFORMATION
# ============================================================

# Same transformation used during tuning
y_log = np.log1p(y)


# ============================================================
# 7. LOGO VALIDATION SETUP
# ============================================================

logo = LeaveOneGroupOut()

n_groups = groups.nunique()


print("=" * 65)
print("        FINAL CATBOOST LOGO EVALUATION")
print("=" * 65)

print(f"Observations:       {len(X)}")
print(f"Organizations:      {n_groups}")
print(f"Features:           {X.shape[1]}")
print(f"Categorical:        {len(cat_indices)}")
print("Validation:         Leave-One-Organization-Out")
print(f"Expected folds:     {n_groups}")


# ============================================================
# 8. FROZEN OPTUNA CONFIGURATION
# ============================================================

# Trial 11 — BEST OPTUNA CONFIGURATION

final_params = {
    'depth': 3,
    'learning_rate': 0.04124173274946173,
    'iterations': 600,
    'l2_leaf_reg': 13.500717070418117,
    'random_strength': 0.41882193959396974,
    'bagging_temperature': 1.9187145799605694
}


print("\n" + "-" * 65)
print("FROZEN OPTUNA PARAMETERS")
print("-" * 65)

for name, value in final_params.items():
    print(f"{name:<22}: {value}")


# ============================================================
# 9. STORAGE FOR OUT-OF-FOLD PREDICTIONS
# ============================================================

all_true = []
all_pred = []

prediction_records = []


# ============================================================
# 10. FULL LOGO CROSS-VALIDATION
# ============================================================

print("\n")
print("Starting final LOGO evaluation...")
print()


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y_log, groups),
    start=1
):

    # --------------------------------------------------------
    # Split data
    # --------------------------------------------------------

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_log = y_log.iloc[train_idx]

    true = y.iloc[test_idx].to_numpy()


    # --------------------------------------------------------
    # Create CatBoost model
    # --------------------------------------------------------

    model = CatBoostRegressor(

        # FROZEN OPTUNA PARAMETERS
        depth=final_params['depth'],
        learning_rate=final_params['learning_rate'],
        iterations=final_params['iterations'],
        l2_leaf_reg=final_params['l2_leaf_reg'],
        random_strength=final_params['random_strength'],
        bagging_temperature=final_params['bagging_temperature'],

        # Model settings
        cat_features=cat_indices,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,

        # Use all available CPU threads
        thread_count=-1,

        # No training output
        verbose=False,

        # Prevent CatBoost from creating extra files
        allow_writing_files=False
    )


    # --------------------------------------------------------
    # Train on all organizations EXCEPT the held-out one
    # --------------------------------------------------------

    model.fit(
        X_train,
        y_train_log
    )


    # --------------------------------------------------------
    # Predict the held-out organization
    # --------------------------------------------------------

    pred_log = model.predict(
        X_test
    )


    # Convert prediction back from log scale
    pred = np.clip(
        np.expm1(pred_log),
        a_min=0,
        a_max=None
    )


    # --------------------------------------------------------
    # Store predictions
    # --------------------------------------------------------

    all_true.extend(true)
    all_pred.extend(pred)


    # --------------------------------------------------------
    # Store detailed project-level results
    # --------------------------------------------------------

    for i, original_idx in enumerate(test_idx):

        actual = true[i]
        predicted = pred[i]

        absolute_error = abs(
            actual - predicted
        )

        mre_value = (
            absolute_error / actual
        )

        prediction_records.append({
            'Fold': fold,
            'Organization_ID': groups.iloc[original_idx],
            'Actual_Effort': actual,
            'Predicted_Effort': predicted,
            'Absolute_Error': absolute_error,
            'MRE': mre_value
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Fold {fold:>2}/{n_groups} completed"
    )


# ============================================================
# 11. CONVERT ALL PREDICTIONS TO ARRAYS
# ============================================================

y_true_arr = np.array(
    all_true
)

y_pred_arr = np.array(
    all_pred
)


# ============================================================
# 12. CALCULATE MRE
# ============================================================

mre = (
    np.abs(
        y_true_arr - y_pred_arr
    )
    / y_true_arr
)


# ============================================================
# 13. FINAL EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(
    y_true_arr,
    y_pred_arr
)

r2 = r2_score(
    y_true_arr,
    y_pred_arr
)

mmre = np.mean(
    mre
)

mdmre = np.median(
    mre
)

pred25 = np.mean(
    mre <= 0.25
)


# ============================================================
# 14. STANDARDIZED ACCURACY (Shepperd & MacDonell, 2012)
#     + SIGNIFICANCE TEST AGAINST RANDOM GUESSING
# ============================================================

# SA compares model MAE against the MAE of random guessing, where
# "random guessing" means: for each project, guess the actual effort
# of a different, randomly chosen project from the same evaluation
# set. This is NOT the same as guessing the mean effort — the mean
# is already a fairly strong naive predictor, so using it as the
# baseline understates the baseline's difficulty and inflates SA.
# The baseline is estimated by averaging many random-guess runs.

n_projects = len(y_true_arr)

rng = np.random.default_rng(RANDOM_SEED)

n_bootstraps = 1000


def random_guess_indices(n, rng):

    idx = rng.integers(0, n, size=n)

    same = idx == np.arange(n)

    while same.any():
        idx[same] = rng.integers(0, n, size=same.sum())
        same = idx == np.arange(n)

    return idx


random_maes = np.empty(n_bootstraps)

for b in range(n_bootstraps):

    guess_idx = random_guess_indices(
        n_projects,
        rng
    )

    random_maes[b] = mean_absolute_error(
        y_true_arr,
        y_true_arr[guess_idx]
    )

random_guess_mae = random_maes.mean()

sa = (
    1 - mae / random_guess_mae
) * 100


# --------------------------------------------------------
# Significance test: is the model significantly better than
# random guessing? Mann-Whitney U test on the absolute-error
# distributions, plus the Â12 effect size.
# --------------------------------------------------------

model_ae = np.abs(
    y_true_arr - y_pred_arr
)

guess_idx = random_guess_indices(
    n_projects,
    rng
)

random_ae = np.abs(
    y_true_arr - y_true_arr[guess_idx]
)

u_stat, p_value = mannwhitneyu(
    model_ae,
    random_ae,
    alternative='less'
)

a12 = u_stat / (
    len(model_ae) * len(random_ae)
)


print("\n" + "=" * 65)
print("       STANDARDIZED ACCURACY (SA)")
print("=" * 65)

print(
    f"Random-guess MAE:  {random_guess_mae:.2f} hours "
    f"(mean of {n_bootstraps} bootstraps)"
)

print(
    f"SA:                {sa:.2f}%"
)

print(
    f"Mann-Whitney p:    {p_value:.4g}  "
    "(model AE < random-guess AE)"
)

print(
    f"Â12 effect size:   {a12:.3f}  "
    "(>0.5 = model beats chance more often than not)"
)


# ============================================================
# 15. SENSITIVITY ANALYSIS — EXCLUDING MRE > 100% OUTLIERS
# ============================================================

low_mre_mask = mre <= 1.0

y_true_clean = y_true_arr[low_mre_mask]
y_pred_clean = y_pred_arr[low_mre_mask]
mre_clean = mre[low_mre_mask]

n_clean = len(y_true_clean)

mae_clean = mean_absolute_error(
    y_true_clean,
    y_pred_clean
)

model_ae_clean = np.abs(
    y_true_clean - y_pred_clean
)

random_maes_clean = np.empty(n_bootstraps)

for b in range(n_bootstraps):

    guess_idx = random_guess_indices(
        n_clean,
        rng
    )

    random_maes_clean[b] = mean_absolute_error(
        y_true_clean,
        y_true_clean[guess_idx]
    )

random_guess_mae_clean = random_maes_clean.mean()

sa_clean = (
    1 - mae_clean / random_guess_mae_clean
) * 100

guess_idx_clean = random_guess_indices(
    n_clean,
    rng
)

random_ae_clean = np.abs(
    y_true_clean - y_true_clean[guess_idx_clean]
)

u_stat_clean, p_value_clean = mannwhitneyu(
    model_ae_clean,
    random_ae_clean,
    alternative='less'
)

a12_clean = u_stat_clean / (
    len(model_ae_clean) * len(random_ae_clean)
)


print("\n" + "=" * 65)
print("       SENSITIVITY ANALYSIS — EXCLUDING MRE > 100%")
print("=" * 65)

print(
    f"Projects evaluated:  {n_clean} / {n_projects}"
)

print(
    f"MAE:                 {mae_clean:.2f} hours"
)

print(
    f"MMRE:                {np.mean(mre_clean):.4f} "
    f"({np.mean(mre_clean) * 100:.2f}%)"
)

print(
    f"MdMRE:               {np.median(mre_clean):.4f} "
    f"({np.median(mre_clean) * 100:.2f}%)"
)

print(
    f"PRED(25):            {np.mean(mre_clean <= 0.25) * 100:.2f}%"
)

print(
    f"SA:                  {sa_clean:.2f}%"
)

print(
    f"Mann-Whitney p:      {p_value_clean:.4g}"
)

print(
    f"Â12 effect size:     {a12_clean:.3f}"
)


# ============================================================
# 16. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("       FINAL CATBOOST LOGO-CV RESULTS")
print("=" * 65)

print(
    f"MAE:       {mae:.2f} hours"
)

print(
    f"R²:        {r2:.4f}"
)

print(
    f"MMRE:      {mmre:.4f} ({mmre * 100:.2f}%)"
)

print(
    f"MdMRE:     {mdmre:.4f} ({mdmre * 100:.2f}%)"
)

print(
    f"PRED(25):  {pred25 * 100:.2f}%"
)

print(
    f"SA:        {sa:.2f}%"
)

print("=" * 65)


# ============================================================
# 17. PROJECT-LEVEL PREDICTION TABLE
# ============================================================

predictions_df = pd.DataFrame(
    prediction_records
)


print("\nProject-level predictions created.")
print(
    f"Number of predictions: {len(predictions_df)}"
)

print("\nFirst 10 predictions:")
print(
    predictions_df.head(10)
)

        FINAL CATBOOST LOGO EVALUATION
Observations:       119
Organizations:      41
Features:           50
Categorical:        6
Validation:         Leave-One-Organization-Out
Expected folds:     41

-----------------------------------------------------------------
FROZEN OPTUNA PARAMETERS
-----------------------------------------------------------------
depth                 : 3
learning_rate         : 0.04124173274946173
iterations            : 600
l2_leaf_reg           : 13.500717070418117
random_strength       : 0.41882193959396974
bagging_temperature   : 1.9187145799605694


Starting final LOGO evaluation...

Fold  1/41 completed
Fold  2/41 completed
Fold  3/41 completed
Fold  4/41 completed
Fold  5/41 completed
Fold  6/41 completed
Fold  7/41 completed
Fold  8/41 completed
Fold  9/41 completed
Fold 10/41 completed
Fold 11/41 completed
Fold 12/41 completed
Fold 13/41 completed
Fold 14/41 completed
Fold 15/41 completed
Fold 16/41 completed
Fold 17/41 completed
Fold 18/41 complete

<h2>Feature selection</h2>


<h3>feature importance </h3>

In [22]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import mannwhitneyu


# ============================================================
# FINAL CATBOOST EVALUATION
# FROZEN OPTUNA CONFIGURATION + FULL LOGO-CV
# ============================================================

RANDOM_SEED = 42


# ============================================================
# 1. COLUMN DEFINITIONS
# ============================================================

target_col = 'Effort_Actual effort'
group_col = 'General Information_Organization id'
id_col = 'General Information_ProjID'


# ============================================================
# 2. DATA PREPARATION
# ============================================================

# Convert target to numeric
y = pd.to_numeric(
    df_clean[target_col],
    errors='coerce'
)

# Remove rows where target is missing
valid_rows = y.notna()

df = df_clean.loc[valid_rows].copy()

y = y.loc[valid_rows].reset_index(drop=True)

groups = (
    df[group_col]
    .reset_index(drop=True)
)


# ============================================================
# 3. FEATURES
# ============================================================

# Target and organization ID must NOT be predictors.
# Project ID is also excluded if present.

cols_to_drop = [
    target_col,
    group_col
]

if id_col in df.columns:
    cols_to_drop.append(id_col)

X = df.drop(
    columns=cols_to_drop
).copy()


# ============================================================
# 4. CATEGORICAL FEATURES
# ============================================================

nominal_cat_cols = [
    'Developers_Team selection',
    'Developers_DBMS  expert availability',  # NOTE: two spaces — matches
                                              # actual column name. The
                                              # original single-space
                                              # version silently failed
                                              # to match and this column
                                              # was being treated as
                                              # numeric instead.
    'General Information_Development type',
    'General Information_Application domain',
    'Environment_Government policy impact',
    'Environment_Developer hiring policy',
    'Environment_Developer training'
]

# Keep only categorical columns that actually exist
nominal_cat_cols = [
    col
    for col in nominal_cat_cols
    if col in X.columns
]


# Convert categorical values to strings
for col in nominal_cat_cols:

    X[col] = (
        X[col]
        .fillna('Missing')
        .astype(str)
        .str.replace(
            r'\.0$',
            '',
            regex=True
        )
    )


# ============================================================
# 5. NUMERICAL FEATURES
# ============================================================

numeric_cols = [
    col
    for col in X.columns
    if col not in nominal_cat_cols
]

for col in numeric_cols:

    X[col] = pd.to_numeric(
        X[col],
        errors='coerce'
    )


# ============================================================
# 5b. ENGINEERED SIZE-PROXY FEATURES
# ============================================================
# The top existing features (Team size, Dedicated team members,
# Estimated duration, Object points) are already strongly
# correlated with effort individually. The likely gap is not a
# missing signal but that effort is naturally a MULTIPLICATIVE
# function of these (roughly: team size x duration x utilization
# ~= person-hours), which a shallow (depth=3) boosted tree can
# only approximate through many splits. Giving it the product
# directly removes that burden.

epsilon = 1e-6

X['Team_Capacity'] = (
    X['Developers_Team size']
    * X['General Information_Estimated  duration']
)

X['Dedicated_Capacity'] = (
    X['Developers_Dedicated team members']
    * X['General Information_Estimated  duration']
)

# What fraction of the team is actually dedicated to this project —
# a signal that's orthogonal to raw team size, since two projects
# with the same team size can have very different dedication levels.
X['Dedication_Ratio'] = (
    X['Developers_Dedicated team members']
    / (X['Developers_Team size'] + epsilon)
)

# Functional size per team member — a rough productivity/staffing
# expectation proxy, distinct from either raw feature alone.
X['Points_per_Person'] = (
    X['Size_Object points']
    / (X['Developers_Team size'] + epsilon)
)

# Log-transform the heavily right-skewed raw size features. Most
# projects (especially the small ones the model overestimates)
# cluster densely in the low range of these features; log-spacing
# gives the shallow trees finer split resolution there instead of
# most of the range being dominated by a few large outlier projects.
skewed_size_cols = [
    'Developers_Team size',
    'Developers_Dedicated team members',
    'Size_Object points',
    'General Information_Estimated  duration'
]

for col in skewed_size_cols:
    X[f'{col}_log'] = np.log1p(
        X[col].clip(lower=0)
    )

engineered_numeric_cols = [
    'Team_Capacity',
    'Dedicated_Capacity',
    'Dedication_Ratio',
    'Points_per_Person'
] + [f'{col}_log' for col in skewed_size_cols]

numeric_cols = numeric_cols + engineered_numeric_cols


# CatBoost categorical feature indices
# (computed AFTER all new columns are added, since positions shift)
cat_indices = [
    X.columns.get_loc(col)
    for col in nominal_cat_cols
]


# ============================================================
# 6. TARGET TRANSFORMATION
# ============================================================

# Same transformation used during tuning
y_log = np.log1p(y)


# ============================================================
# 7. LOGO VALIDATION SETUP
# ============================================================

logo = LeaveOneGroupOut()

n_groups = groups.nunique()


print("=" * 65)
print("        FINAL CATBOOST LOGO EVALUATION")
print("=" * 65)

print(f"Observations:       {len(X)}")
print(f"Organizations:      {n_groups}")
print(f"Features:           {X.shape[1]}")
print(f"Categorical:        {len(cat_indices)}")
print("Validation:         Leave-One-Organization-Out")
print(f"Expected folds:     {n_groups}")


# ============================================================
# 8. FROZEN OPTUNA CONFIGURATION
# ============================================================

# Trial 11 — BEST OPTUNA CONFIGURATION

final_params = {
    'depth': 3,
    'learning_rate': 0.04124173274946173,
    'iterations': 600,
    'l2_leaf_reg': 13.500717070418117,
    'random_strength': 0.41882193959396974,
    'bagging_temperature': 1.9187145799605694
}


print("\n" + "-" * 65)
print("FROZEN OPTUNA PARAMETERS")
print("-" * 65)

for name, value in final_params.items():
    print(f"{name:<22}: {value}")


# ============================================================
# 9. STORAGE FOR OUT-OF-FOLD PREDICTIONS
# ============================================================

all_true = []
all_pred = []

prediction_records = []


# ============================================================
# 10. FULL LOGO CROSS-VALIDATION
# ============================================================

print("\n")
print("Starting final LOGO evaluation...")
print()


for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y_log, groups),
    start=1
):

    # --------------------------------------------------------
    # Split data
    # --------------------------------------------------------

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train_log = y_log.iloc[train_idx]

    true = y.iloc[test_idx].to_numpy()


    # --------------------------------------------------------
    # Create CatBoost model
    # --------------------------------------------------------

    model = CatBoostRegressor(

        # FROZEN OPTUNA PARAMETERS
        depth=final_params['depth'],
        learning_rate=final_params['learning_rate'],
        iterations=final_params['iterations'],
        l2_leaf_reg=final_params['l2_leaf_reg'],
        random_strength=final_params['random_strength'],
        bagging_temperature=final_params['bagging_temperature'],

        # Model settings
        cat_features=cat_indices,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,

        # Use all available CPU threads
        thread_count=-1,

        # No training output
        verbose=False,

        # Prevent CatBoost from creating extra files
        allow_writing_files=False
    )


    # --------------------------------------------------------
    # Train on all organizations EXCEPT the held-out one
    # --------------------------------------------------------

    model.fit(
        X_train,
        y_train_log
    )


    # --------------------------------------------------------
    # Predict the held-out organization
    # --------------------------------------------------------

    pred_log = model.predict(
        X_test
    )


    # Convert prediction back from log scale
    pred = np.clip(
        np.expm1(pred_log),
        a_min=0,
        a_max=None
    )


    # --------------------------------------------------------
    # Store predictions
    # --------------------------------------------------------

    all_true.extend(true)
    all_pred.extend(pred)


    # --------------------------------------------------------
    # Store detailed project-level results
    # --------------------------------------------------------

    for i, original_idx in enumerate(test_idx):

        actual = true[i]
        predicted = pred[i]

        absolute_error = abs(
            actual - predicted
        )

        mre_value = (
            absolute_error / actual
        )

        prediction_records.append({
            'Fold': fold,
            'Organization_ID': groups.iloc[original_idx],
            'Actual_Effort': actual,
            'Predicted_Effort': predicted,
            'Absolute_Error': absolute_error,
            'MRE': mre_value
        })


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    print(
        f"Fold {fold:>2}/{n_groups} completed"
    )


# ============================================================
# 11. CONVERT ALL PREDICTIONS TO ARRAYS
# ============================================================

y_true_arr = np.array(
    all_true
)

y_pred_arr = np.array(
    all_pred
)


# ============================================================
# 12. CALCULATE MRE
# ============================================================

mre = (
    np.abs(
        y_true_arr - y_pred_arr
    )
    / y_true_arr
)


# ============================================================
# 13. FINAL EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(
    y_true_arr,
    y_pred_arr
)

r2 = r2_score(
    y_true_arr,
    y_pred_arr
)

mmre = np.mean(
    mre
)

mdmre = np.median(
    mre
)

pred25 = np.mean(
    mre <= 0.25
)


# ============================================================
# 14a. CORRELATION OF EXISTING NUMERIC FEATURES WITH EFFORT
# ============================================================
# Purpose: find out whether any existing column is already a
# decent size proxy (high correlation with actual effort) that
# just isn't being weighted heavily, versus a genuine gap where
# NO column captures project scale well.

numeric_corr = (
    X[numeric_cols]
    .assign(Actual_Effort=y.values)
    .corr(numeric_only=True)['Actual_Effort']
    .drop('Actual_Effort')
    .sort_values(ascending=False)
)

print("\n" + "=" * 65)
print("       NUMERIC FEATURE CORRELATION WITH ACTUAL EFFORT")
print("=" * 65)
print(numeric_corr.to_string())


# ============================================================
# 14b. FEATURE IMPORTANCE — WHAT IS THE MODEL USING AS A
#      PROJECT-SIZE SIGNAL RIGHT NOW?
# ============================================================
# Retrain once on the FULL dataset (no held-out org) purely to
# get a stable, whole-dataset importance ranking. This model is
# NOT used for evaluation — LOGO-CV results above are unaffected.

full_model = CatBoostRegressor(
    depth=final_params['depth'],
    learning_rate=final_params['learning_rate'],
    iterations=final_params['iterations'],
    l2_leaf_reg=final_params['l2_leaf_reg'],
    random_strength=final_params['random_strength'],
    bagging_temperature=final_params['bagging_temperature'],
    cat_features=cat_indices,
    loss_function='RMSE',
    random_seed=RANDOM_SEED,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

full_model.fit(X, y_log)

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': full_model.get_feature_importance()
}).sort_values('Importance', ascending=False)

print("\n" + "=" * 65)
print("       FEATURE IMPORTANCE (full-data fit)")
print("=" * 65)
print(importances.head(20).to_string(index=False))

# Flag features with high importance but weak direct correlation —
# a common signature of a feature acting as a proxy/fingerprint for
# something else (e.g. organization identity) rather than a genuine
# effort driver. This matters specifically for LOGO-CV, where the
# model must generalize to organizations it has never seen.
suspect = importances.merge(
    numeric_corr.rename('Correlation').reset_index().rename(
        columns={'index': 'Feature'}
    ),
    on='Feature',
    how='left'
)
suspect['Abs_Correlation'] = suspect['Correlation'].abs()
suspect_flagged = suspect[
    (suspect['Importance'] > 3.0) & (suspect['Abs_Correlation'] < 0.10)
]

if len(suspect_flagged) > 0:
    print("\n" + "-" * 65)
    print("FLAGGED: high importance, weak direct correlation")
    print("(possible proxy for organization identity / overfit risk)")
    print("-" * 65)
    print(
        suspect_flagged[['Feature', 'Importance', 'Correlation']]
        .to_string(index=False)
    )


# ============================================================
# 14c. FULL COLUMN LIST — FOR MANUAL REVIEW
# ============================================================
# Print every available column so size-proxy candidates that
# exist in the raw data but aren't strongly used can be spotted
# by eye (e.g. team size, duration, functional size, requirements
# count -- naming depends on the source dataset).

print("\n" + "=" * 65)
print("       ALL AVAILABLE FEATURE COLUMNS")
print("=" * 65)
for col in X.columns:
    tag = 'categorical' if col in nominal_cat_cols else 'numeric'
    print(f"  [{tag:<11}] {col}")


# ============================================================
# 15. STANDARDIZED ACCURACY (Shepperd & MacDonell, 2012)
#     + SIGNIFICANCE TEST AGAINST RANDOM GUESSING
# ============================================================

# SA compares model MAE against the MAE of random guessing, where
# "random guessing" means: for each project, guess the actual effort
# of a different, randomly chosen project from the same evaluation
# set. This is NOT the same as guessing the mean effort — the mean
# is already a fairly strong naive predictor, so using it as the
# baseline understates the baseline's difficulty and inflates SA.
# The baseline is estimated by averaging many random-guess runs.

n_projects = len(y_true_arr)

rng = np.random.default_rng(RANDOM_SEED)

n_bootstraps = 1000


def random_guess_indices(n, rng):

    idx = rng.integers(0, n, size=n)

    same = idx == np.arange(n)

    while same.any():
        idx[same] = rng.integers(0, n, size=same.sum())
        same = idx == np.arange(n)

    return idx


random_maes = np.empty(n_bootstraps)

for b in range(n_bootstraps):

    guess_idx = random_guess_indices(
        n_projects,
        rng
    )

    random_maes[b] = mean_absolute_error(
        y_true_arr,
        y_true_arr[guess_idx]
    )

random_guess_mae = random_maes.mean()

sa = (
    1 - mae / random_guess_mae
) * 100


# --------------------------------------------------------
# Significance test: is the model significantly better than
# random guessing? Mann-Whitney U test on the absolute-error
# distributions, plus the Â12 effect size.
# --------------------------------------------------------

model_ae = np.abs(
    y_true_arr - y_pred_arr
)

guess_idx = random_guess_indices(
    n_projects,
    rng
)

random_ae = np.abs(
    y_true_arr - y_true_arr[guess_idx]
)

u_stat, p_value = mannwhitneyu(
    model_ae,
    random_ae,
    alternative='less'
)

# mannwhitneyu(x, y, ...) always returns the U statistic FOR x,
# i.e. U_x = P(x > y) * n1 * n2 (with 0.5 credit for ties).
# Â12 here should mean "probability the model's error is SMALLER
# than a random guess's error", i.e. P(model_ae < random_ae).
# u_stat / (n1*n2) is P(model_ae > random_ae) -- the opposite --
# so it must be inverted.
a12 = 1 - (
    u_stat / (len(model_ae) * len(random_ae))
)


print("\n" + "=" * 65)
print("       STANDARDIZED ACCURACY (SA)")
print("=" * 65)

print(
    f"Random-guess MAE:  {random_guess_mae:.2f} hours "
    f"(mean of {n_bootstraps} bootstraps)"
)

print(
    f"SA:                {sa:.2f}%"
)

print(
    f"Mann-Whitney p:    {p_value:.4g}  "
    "(model AE < random-guess AE)"
)

print(
    f"Â12 effect size:   {a12:.3f}  "
    "(P(model error < random-guess error); >0.5 = model beats chance)"
)


# ============================================================
# 16. SENSITIVITY ANALYSIS — EXCLUDING MRE > 100% OUTLIERS
# ============================================================

low_mre_mask = mre <= 1.0

y_true_clean = y_true_arr[low_mre_mask]
y_pred_clean = y_pred_arr[low_mre_mask]
mre_clean = mre[low_mre_mask]

n_clean = len(y_true_clean)

mae_clean = mean_absolute_error(
    y_true_clean,
    y_pred_clean
)

model_ae_clean = np.abs(
    y_true_clean - y_pred_clean
)

random_maes_clean = np.empty(n_bootstraps)

for b in range(n_bootstraps):

    guess_idx = random_guess_indices(
        n_clean,
        rng
    )

    random_maes_clean[b] = mean_absolute_error(
        y_true_clean,
        y_true_clean[guess_idx]
    )

random_guess_mae_clean = random_maes_clean.mean()

sa_clean = (
    1 - mae_clean / random_guess_mae_clean
) * 100

guess_idx_clean = random_guess_indices(
    n_clean,
    rng
)

random_ae_clean = np.abs(
    y_true_clean - y_true_clean[guess_idx_clean]
)

u_stat_clean, p_value_clean = mannwhitneyu(
    model_ae_clean,
    random_ae_clean,
    alternative='less'
)

# Same correction as above: invert to get P(model_ae < random_ae).
a12_clean = 1 - (
    u_stat_clean / (len(model_ae_clean) * len(random_ae_clean))
)


print("\n" + "=" * 65)
print("       SENSITIVITY ANALYSIS — EXCLUDING MRE > 100%")
print("=" * 65)

print(
    f"Projects evaluated:  {n_clean} / {n_projects}"
)

print(
    f"MAE:                 {mae_clean:.2f} hours"
)

print(
    f"MMRE:                {np.mean(mre_clean):.4f} "
    f"({np.mean(mre_clean) * 100:.2f}%)"
)

print(
    f"MdMRE:               {np.median(mre_clean):.4f} "
    f"({np.median(mre_clean) * 100:.2f}%)"
)

print(
    f"PRED(25):            {np.mean(mre_clean <= 0.25) * 100:.2f}%"
)

print(
    f"SA:                  {sa_clean:.2f}%"
)

print(
    f"Mann-Whitney p:      {p_value_clean:.4g}"
)

print(
    f"Â12 effect size:     {a12_clean:.3f}"
)


# ============================================================
# 17. FINAL RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("       FINAL CATBOOST LOGO-CV RESULTS")
print("=" * 65)

print(
    f"MAE:       {mae:.2f} hours"
)

print(
    f"R²:        {r2:.4f}"
)

print(
    f"MMRE:      {mmre:.4f} ({mmre * 100:.2f}%)"
)

print(
    f"MdMRE:     {mdmre:.4f} ({mdmre * 100:.2f}%)"
)

print(
    f"PRED(25):  {pred25 * 100:.2f}%"
)

print(
    f"SA:        {sa:.2f}%"
)

print("=" * 65)


# ============================================================
# 18. PROJECT-LEVEL PREDICTION TABLE
# ============================================================

predictions_df = pd.DataFrame(
    prediction_records
)


print("\nProject-level predictions created.")
print(
    f"Number of predictions: {len(predictions_df)}"
)

print("\nFirst 10 predictions:")
print(
    predictions_df.head(10)
)

        FINAL CATBOOST LOGO EVALUATION
Observations:       119
Organizations:      41
Features:           58
Categorical:        7
Validation:         Leave-One-Organization-Out
Expected folds:     41

-----------------------------------------------------------------
FROZEN OPTUNA PARAMETERS
-----------------------------------------------------------------
depth                 : 3
learning_rate         : 0.04124173274946173
iterations            : 600
l2_leaf_reg           : 13.500717070418117
random_strength       : 0.41882193959396974
bagging_temperature   : 1.9187145799605694


Starting final LOGO evaluation...

Fold  1/41 completed
Fold  2/41 completed
Fold  3/41 completed
Fold  4/41 completed
Fold  5/41 completed
Fold  6/41 completed
Fold  7/41 completed
Fold  8/41 completed
Fold  9/41 completed
Fold 10/41 completed
Fold 11/41 completed
Fold 12/41 completed
Fold 13/41 completed
Fold 14/41 completed
Fold 15/41 completed
Fold 16/41 completed
Fold 17/41 completed
Fold 18/41 complete

In [26]:
# ==========================================
# QUICK EXPERIMENT — TOP N FEATURES ONLY
# ==========================================

results = {}

for top_n in [5, 10, 15, 20, 30, 50]:


    keep = X.columns[importances.head(top_n).index].tolist()
    fold_cat_cols    = [c for c in nominal_cat_cols if c in keep]
    fold_cat_indices = [keep.index(c) for c in fold_cat_cols]

    logo_preds  = []
    logo_true   = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y_log, groups)):

        X_train = X[keep].iloc[train_idx].copy()
        X_test  = X[keep].iloc[test_idx].copy()
        y_train_log = y_log.iloc[train_idx]

        imp = SimpleImputer(strategy='median')
        num_keep = [c for c in keep if c in numeric_cols]
        X_train[num_keep] = imp.fit_transform(X_train[num_keep])
        X_test[num_keep]  = imp.transform(X_test[num_keep])

        for col in fold_cat_cols:
            X_train[col] = X_train[col].fillna('missing')
            X_test[col]  = X_test[col].fillna('missing')

        m = CatBoostRegressor(
            depth=3,
            learning_rate=0.04124173274946173,
            iterations=600,
            l2_leaf_reg=13.5,
            random_strength=0.41882193959396974,
            bagging_temperature=1.9187145799605694,
            cat_features=fold_cat_indices,
            loss_function='RMSE',
            random_seed=42,
            verbose=0
        )
        m.fit(X_train, y_train_log)
        p = np.clip(np.expm1(m.predict(X_test)), 0, None)
        logo_preds.extend(p.tolist())
        logo_true.extend(y.iloc[test_idx].tolist())

    yt = np.array(logo_true)
    yp = np.array(logo_preds)
    v  = yt > 0
    mr = np.abs(yt[v] - yp[v]) / yt[v]
    rg = np.mean(np.abs(yt - np.mean(yt)))
    sa = (1 - mean_absolute_error(yt, yp) / rg) * 100

    results[top_n] = {
        'MMRE':    np.mean(mr),
        'MdMRE':   np.median(mr),
        'PRED25':  np.mean(mr <= 0.25),
        'SA':      sa
    }
    print(f"Top {top_n:2d} features — MMRE={np.mean(mr)*100:.1f}%  "
          f"PRED25={np.mean(mr<=0.25)*100:.1f}%  SA={sa:.1f}%")

# Summary table
print("\n=================================================================")
print("     FEATURE COUNT COMPARISON")
print("=================================================================")
print(f"{'N features':<12} {'MMRE':>8} {'MdMRE':>8} {'PRED(25)':>10} {'SA':>8}")
for n, r in results.items():
    print(f"{n:<12} {r['MMRE']*100:>7.1f}% {r['MdMRE']*100:>7.1f}% "
          f"{r['PRED25']*100:>9.1f}% {r['SA']:>7.1f}%")
          

Top  5 features — MMRE=48.2%  PRED25=36.1%  SA=47.6%
Top 10 features — MMRE=40.4%  PRED25=46.2%  SA=52.7%
Top 15 features — MMRE=39.8%  PRED25=40.3%  SA=54.0%
Top 20 features — MMRE=40.5%  PRED25=40.3%  SA=54.4%
Top 30 features — MMRE=44.1%  PRED25=37.8%  SA=50.9%
Top 50 features — MMRE=45.5%  PRED25=38.7%  SA=48.8%

     FEATURE COUNT COMPARISON
N features       MMRE    MdMRE   PRED(25)       SA
5               48.2%    37.6%      36.1%    47.6%
10              40.4%    26.7%      46.2%    52.7%
15              39.8%    33.4%      40.3%    54.0%
20              40.5%    30.9%      40.3%    54.4%
30              44.1%    36.7%      37.8%    50.9%
50              45.5%    36.4%      38.7%    48.8%


<h2>Final Model with top 15 features</h2>

In [29]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score

# ============================================================
# 1. PREPARE TOP 15 FEATURE SUBSET
# ============================================================

# Isolate top 15 feature names from feature importance rankings
TOP_N = 15
top_15_features = X.columns[importances.head(15).index].tolist()

X_15 = X[top_15_features].copy()

# Map categorical features and positions within top 15 subset
nominal_15 = [col for col in nominal_cat_cols if col in top_15_features]
cat_indices_15 = [top_15_features.index(col) for col in nominal_15]

print("==========================================================")
print(f"RUNNING FINAL MODEL (TOP {TOP_N} FEATURES + DUAN'S SMEARING)")
print("==========================================================")
print(f"Selected Features: {len(top_15_features)}")
print(f"Categorical Features: {len(cat_indices_15)}")


# ============================================================
# 2. LOGO-CV WITH DUAN'S SMEARING FACTOR BIAS CORRECTION
# ============================================================

oof_true = []
oof_pred = []
prediction_records = []

logo = LeaveOneGroupOut()

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_15, y_log, groups), 
    start=1
):
    # Train / Test split
    X_train, X_test = X_15.iloc[train_idx], X_15.iloc[test_idx]
    y_train_log = y_log.iloc[train_idx]
    true = y.iloc[test_idx].to_numpy()

    # Train CatBoost on frozen Optuna parameters
    model = CatBoostRegressor(
        depth=final_params['depth'],
        learning_rate=final_params['learning_rate'],
        iterations=final_params['iterations'],
        l2_leaf_reg=final_params['l2_leaf_reg'],
        random_strength=final_params['random_strength'],
        bagging_temperature=final_params['bagging_temperature'],
        cat_features=cat_indices_15,
        loss_function='RMSE',
        random_seed=RANDOM_SEED,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False
    )

    model.fit(X_train, y_train_log)

    # --- DUAN'S SMEARING FACTOR CALCULATION ---
    # Compute in-sample log residuals on training set
    train_log_preds = model.predict(X_train)
    log_residuals = y_train_log - train_log_preds
    
    # Smearing multiplier: w = mean(exp(residuals))
    smearing_factor = np.mean(np.exp(log_residuals))

    # --- PREDICTION WITH BIAS CORRECTION ---
    test_pred_log = model.predict(X_test)
    
    # Apply smearing correction back to raw scale: w * exp(pred_log) - 1
    pred_raw = smearing_factor * np.exp(test_pred_log) - 1
    pred = np.clip(pred_raw, a_min=0, a_max=None)

    oof_true.extend(true)
    oof_pred.extend(pred)

    for i, orig_idx in enumerate(test_idx):
        prediction_records.append({
            'Fold': fold,
            'Organization_ID': groups.iloc[orig_idx],
            'Actual_Effort': true[i],
            'Predicted_Effort': pred[i],
            'Smearing_Factor': smearing_factor
        })


# ============================================================
# 3. EVALUATION METRICS
# ============================================================

yt = np.array(oof_true)
yp = np.array(oof_pred)

mre = np.abs(yt - yp) / yt
mmre = np.mean(mre)
mdmre = np.median(mre)
pred25 = np.mean(mre <= 0.25)
mae = mean_absolute_error(yt, yp)
r2 = r2_score(yt, yp)

mean_baseline_err = np.mean(np.abs(yt - np.mean(yt)))
sa = (1 - (mae / mean_baseline_err)) * 100

print("\n" + "=" * 65)
print("     FINAL MODEL OUT-OF-FOLD PERFORMANCE METRICS")
print("=" * 65)
print(f"MMRE:      {mmre:.4f} ({mmre * 100:.2f}%)")
print(f"MdMRE:     {mdmre:.4f} ({mdmre * 100:.2f}%)")
print(f"PRED(25):  {pred25 * 100:.2f}%")
print(f"SA Score:  {sa:.2f}%")
print(f"MAE:       {mae:.2f} hours")
print(f"R²:        {r2:.4f}")
print("=" * 65)


# ============================================================
# 4. TRAIN PRODUCTION MODEL ON FULL DATASET
# ============================================================

final_production_model = CatBoostRegressor(
    depth=final_params['depth'],
    learning_rate=final_params['learning_rate'],
    iterations=final_params['iterations'],
    l2_leaf_reg=final_params['l2_leaf_reg'],
    random_strength=final_params['random_strength'],
    bagging_temperature=final_params['bagging_temperature'],
    cat_features=cat_indices_15,
    loss_function='RMSE',
    random_seed=RANDOM_SEED,
    thread_count=-1,
    verbose=False,
    allow_writing_files=False
)

# Fit on all available historical data
final_production_model.fit(X_15, y_log)

# Calculate global smearing factor for deployment
global_log_residuals = y_log - final_production_model.predict(X_15)
global_smearing_factor = np.mean(np.exp(global_log_residuals))

print(f"\nProduction model fitted successfully.")
print(f"Global Deployment Smearing Factor: {global_smearing_factor:.4f}")

# Save deployment artifact
final_production_model.save_model("catboost_effort_top15_final.cbm")

RUNNING FINAL MODEL (TOP 15 FEATURES + DUAN'S SMEARING)
Selected Features: 15
Categorical Features: 0

     FINAL MODEL OUT-OF-FOLD PERFORMANCE METRICS
MMRE:      0.4079 (40.79%)
MdMRE:     0.3039 (30.39%)
PRED(25):  41.18%
SA Score:  53.64%
MAE:       3643.90 hours
R²:        0.6344

Production model fitted successfully.
Global Deployment Smearing Factor: 1.0271
